In [58]:
import re
import requests
import pandas as pd

API_KEY = "SEQpLCXd2XnJNiEtRyTTGqg0GECmOQcmSgGoI5KLOXt3p8g4"
YEAR    = 1992
MONTH   = 6

import necessary packages and API Key from NY Developer Times site.

In [59]:
def fetch_articles(year: int, month: int, api_key: str) -> list[dict]:
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    params = {"api-key": api_key}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data["response"]["docs"]

In [60]:
articles = fetch_articles(1992, 6, API_KEY)
print(len(articles))
print(articles[0])
print(df["headline"][100])
print(df["abstract"][100])
print(df["url"][100])
sports = df[df["section"] == "Sports"]
print(sports["headline"].head(10))
print(df["section"].unique())

7123
{'abstract': '*3*** COMPANY REPORTS **', 'web_url': 'https://www.nytimes.com/1992/06/01/business/abrams-industries-reports-earnings-for-qtr-to-april-30.html', 'snippet': '', 'lead_paragraph': '', 'print_section': 'D', 'print_page': '4', 'source': 'The New York Times', 'multimedia': [], 'headline': {'main': 'Abrams Industries reports earnings for Qtr to April 30', 'kicker': None, 'content_kicker': None, 'print_headline': 'Abrams Industries reports earnings for Qtr to April 30', 'name': None, 'seo': None, 'sub': None}, 'keywords': [{'name': 'subject', 'value': 'COMPANY EARNINGS', 'rank': 1, 'major': 'N'}], 'pub_date': '1992-06-01T05:00:00+0000', 'document_type': 'article', 'news_desk': 'Financial Desk', 'section_name': 'Business Day', 'byline': {'original': '', 'person': [], 'organization': None}, 'type_of_material': 'Statistics', '_id': 'nyt://article/01e31eab-0eda-5061-9b76-ccf6ce3ea8b1', 'word_count': 53, 'uri': 'nyt://article/01e31eab-0eda-5061-9b76-ccf6ce3ea8b1'}
Pittsburgh Str

Aquire/fetch data and store in variable "articles". I chose my birthday month and year for the dates. Use JSON to read data. If request is unsuccessful, print error msg. Test if function works by printing some data.

In [61]:
def extract_author(byline_raw: str) -> str:

    if not byline_raw:
        return "Unknown"
    match = re.search(r"By ([A-Z][A-Z\s\-]+?)(?:\s+and\s+|$)", byline_raw, re.IGNORECASE)
    return match.group(1).strip().title() if match else "Unknown"

get Author by capturing everything after "by" and "and". Turn into string. Return error msg if can't find name.

In [62]:
def extract_year_month(pub_date: str) -> tuple[str, str]:
    match = re.search(r"(\d{4})-(\d{2})-\d{2}", pub_date)
    if match:
        return match.group(1), match.group(2)
    return ("", "")

This function looks for date pattern YYYY-MM-DD. group (1) is year 4 digits group (2) is month 2 digits. If nothing found, return empty date.

In [63]:
def extract_section_from_url(url: str) -> str:
    match = re.search(r"nytimes\.com/\d{4}/\d{2}/\d{2}/([a-z\-]+)/", url)
    return match.group(1) if match else "unknown"

Get article from NY Times site that matches date.

In [64]:
def clean_abstract(text: str) -> str:
    if not text:
        return ""
    no_tags = re.sub(r"<[^>]+>", "", text)
    clean   = re.sub(r"\s{2,}", " ", no_tags).strip()
    return clean

Clean up empty spaces, html tags.

In [65]:
def articles_to_dataframe(articles: list[dict]) -> pd.DataFrame:
    rows = []

    for doc in articles:

        headline    = doc.get("headline", {}).get("main", "")
        byline_raw  = doc.get("byline",   {}).get("original", "") or ""
        pub_date    = doc.get("pub_date",  "")
        web_url     = doc.get("web_url",   "")
        abstract    = doc.get("abstract",  "") or ""
        section     = doc.get("section_name", "") or ""
        word_count  = doc.get("word_count", 0)


        author          = extract_author(byline_raw)
        year, month_str = extract_year_month(pub_date)
        url_section     = extract_section_from_url(web_url)
        clean_abs       = clean_abstract(abstract)

        rows.append({
            "headline"    : headline,
            "author"      : author,
            "pub_year"    : year,
            "pub_month"   : month_str,
            "section"     : section or url_section,
            "abstract"    : clean_abs,
            "word_count"  : word_count,
            "url"         : web_url,
        })

    df = pd.DataFrame(rows)


    df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce").fillna(0).astype(int)
    df["pub_year"]   = pd.to_numeric(df["pub_year"],   errors="coerce")

    return df

organize fetched articles by Author, headline, year, month, section, abstrat, word count, and url. store in panda dataframe.

In [66]:
def main():
    print(f"Fetching NYT Archive: {YEAR}-{MONTH:02d} …")
    articles = fetch_articles(YEAR, MONTH, API_KEY)
    print(f"  → {len(articles)} articles retrieved")

    df = articles_to_dataframe(articles)

    print("\n--- DataFrame Info ---")
    print(df.info())

    print("\n--- First 5 rows ---")
    print(df.head())

    print("\n--- Articles per section (top 10) ---")
    print(df["section"].value_counts().head(10))

    print("\n--- Average word count by section ---")
    print(df.groupby("section")["word_count"].mean().sort_values(ascending=False).head(10))


    output_file = f"nyt_archive_{YEAR}_{MONTH:02d}.csv"
    df.to_csv(output_file, index=False)
    print(f"\nSaved to {output_file}")


    return df


In [67]:
    df = main()

Fetching NYT Archive: 1992-06 …
  → 7123 articles retrieved

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7123 entries, 0 to 7122
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   headline    7123 non-null   object
 1   author      7123 non-null   object
 2   pub_year    7123 non-null   int64 
 3   pub_month   7123 non-null   object
 4   section     7123 non-null   object
 5   abstract    7123 non-null   object
 6   word_count  7123 non-null   int64 
 7   url         7123 non-null   object
dtypes: int64(2), object(6)
memory usage: 445.3+ KB
None

--- First 5 rows ---
                                            headline   author  pub_year  \
0  Abrams Industries reports earnings for Qtr to ...  Unknown      1992   
1  Nu Horizons Electronics reports earnings for Q...  Unknown      1992   
2  Datamark Inc. reports earnings for Qtr to Marc...  Unknown      1992   
3                            Eq

call our main function. print everything that we did.